In [1]:
from __future__ import annotations

import os
import base64
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path
from retreival import retrieval
from index_embedd import save,VectorStore
from data_gathering import ingest
from LLM import  initialize_hf_llm
from index_embedd import initialize
import magic


# Force Hugging Face to look directly at your D drive directory bypassing the link
os.environ["HF_HOME"] = r"D:\models\huggingface"
os.environ["TORCH_HOME"] = r"D:\models\torch_models"



In [2]:
load_dotenv()
hf_api_key = os.getenv("HUGGIN_FACE_API")

client = OpenAI(
    api_key=hf_api_key,
    base_url="https://router.huggingface.co/v1"
)

model_name = "Qwen/Qwen3-4B-Instruct-2507"
vision_model_name = "Qwen/Qwen3-VL-8B-Instruct"

# Convert local image file to base64 string
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")




In [ ]:
##test#######
response = client.chat.completions.create(
    model= model_name,
    messages= [
        {
            "role":"user", "content":"give me a simple code for rag using langchain"
        }
    ]
)

# print(response.choices[0].message.content)
display(Markdown(response.choices[0].message.content))



local_image_path = r"C:\Users\shahin\Desktop\pics\er.jfif"
base64_image = encode_image_to_base64(local_image_path)
image_data_url = f"data:image/jpeg;base64,{base64_image}"

# Request execution block
response = client.chat.completions.create(
    model=vision_model_name,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what you see in this picture in detail."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data_url  # Passes the parsed base64 data string variable
                    }
                }
            ]
        }
    ]
)

display(Markdown(response.choices[0].message.content))

In [4]:

def detect_file_type(file_path):
    path = Path(file_path)

    suffix, mime = None, None

    if path.exists():
        suffix = path.suffix
        mime = magic.from_file(str(path), mime=True)

    return mime, suffix, path

def route_file(mime, suffix, path):
    if mime == "application/pdf":
        #pdf
        handle_pdf(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        "application/msword"
    ]:
        #word
        handle_word(path)


    if mime in [
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        "application/vnd.ms-excel"
    ]:
        handle_excel(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        "application/vnd.ms-powerpoint"
    ]:

        handle_pp(path)

    # if mime.startswith("image/"):
    #     return "Image"
    #
    # if mime.startswith("audio/"):
    #     return "Audio"
    #
    # if mime.startswith("video/"):
    #     return "Video"

    return "invalid"







###########Alternative using model###################
# print(mime, suffix)
# system_prompt = """
# Your role is just to analyse the MIME and the suffix passed to you and detect the file type.
# You must identify if it is Excel, Word document, Powerpoint, Audio, Video, Image or PDF.
# Just return one word.
# If what is provided to you is not valid just return the word: invalid.
# """
#
# response = client.chat.completions.create(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {
#             "role": "user",
#             "content": f"MIME: {mime}, suffix: {suffix}"
#         }
#     ]
# )
#
# print(response.choices[0].message.content)

In [5]:
from word_handler import process_docx


def handle_word(path: str):
    chunks, doc_meta, img_to_ch, tbl_to_ch = process_docx(path)

    if doc_meta.doc_id in VectorStore.indexed_docs:
        print(f"⏭️  skipped {doc_meta.doc_id} (already indexed)")
        return

    ingest(chunks, doc_meta, img_to_ch, tbl_to_ch)
    VectorStore.index_chunks(chunks)
    VectorStore.index_images(chunks)
    VectorStore.indexed_docs.add(doc_meta.doc_id)

def handle_excel(path):
    pass

def handle_pp(path):
    pass


def handle_pdf(path):
    pass

In [6]:
import sys

if __name__ == "__main__":
    initialize()
    initialize_hf_llm()
    mime, suffix, path = detect_file_type(r"F:\university\az e riz\گزارش.docx")
    route_file(mime, suffix, path)

    results = retrieval(query="آخرین بخش آزمایش")
    print("final results")
    current_direct = os.curdir
    save(current_direct)
    # for r in results:
    #     print(r)
    #context = build_context(results, [])
    #print(context[:600])


⚠️  Low confidence — activating query expansion fallback...
   Expanded query: 'آخرین بخش آزمایش, بخش پایانی, بخش آخر, قسمت نهایی, مراحل نهایی, بخش دوم'
⚠️  Low image confidence — activating query expansion fallback...
[]
final results
saved to ./
